# Reproduce DeepSequence paper findings

Fast path: **load locked JSON artifacts** under `ab_runs/` and rebuild primary tables/figures.
Does **not** retrain the locked 800 panel by default.

**Primary protocol = Direct-MH** (PAPER.md §3.10 / §5). Recursive one-step rollout is **appendix-only** (Appendix E).

| Claim | Artifact / figure |
|-------|-------------------|
| Table 1 daily Direct-MH IWMAE+CumMAE | `daily_direct_mh60_locked800_s42.json` · Figs **D1–D2** |
| Table D-S1 daily zones | `strata_daily_direct_s42.json` · Fig **D3** |
| Tables Z / W / L weekly + grain | weekly + zero-rate JSON · Figs **W1–W3** |
| Tables W-S1 / W-S2 weekly zones | `strata_weekly_direct_s42.json` · Fig **W6** |
| Architecture | Fig **m5** (optional regen) |
| Weekly qualitative W4–W5 | optional / long (needs data) |
| Recursive / multi-seed | Appendix E only |

## Environment
- Python: `.venv-test`
- `TF_USE_LEGACY_KERAS=1` (required for this stack)
- Enterprise daily panel: `DEEPSEQUENCE_DATA_DIR` → Jubilant `data/` (not shipped; only for optional long cells)
- Package imports only — no `examples/*.py` library modules

In [ ]:
import os
from pathlib import Path

# Prefer legacy Keras before TF imports elsewhere
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")

ROOT = Path.cwd()
if not (ROOT / "PAPER.md").exists():
    # notebook may be opened with cwd = examples/
    if (ROOT.parent / "PAPER.md").exists():
        ROOT = ROOT.parent
        os.chdir(ROOT)

print("repo", ROOT)
print("TF_USE_LEGACY_KERAS", os.environ.get("TF_USE_LEGACY_KERAS"))
print("DEEPSEQUENCE_DATA_DIR", os.environ.get("DEEPSEQUENCE_DATA_DIR", "<unset>"))

## Load artifact helpers (package)

`deepsequence_hierarchical_attention.eval.paper_artifacts` reads locked JSON and builds Direct-MH / strata tables.

In [ ]:
import pandas as pd
from IPython.display import Image, display, Markdown

from deepsequence_hierarchical_attention.eval.paper_artifacts import (
    DEFAULT_PATHS,
    PRIMARY_FIGURES,
    bakeoff_table,
    figure_path,
    format_markdown_table,
    like_for_like_direct_table,
    load_json,
    multiseed_iwmae_pivot,
    recursive_bakeoff_table,
    strata_table,
    zero_rate_summary,
)


def show_figs(*labels: str) -> None:
    for lab in labels:
        p = figure_path(lab)
        display(Markdown(f"**Figure {lab}** — `{PRIMARY_FIGURES[lab]}`"))
        if p.exists():
            display(Image(filename=str(p)))
        else:
            print(f"missing: {p}")


zr = zero_rate_summary()
zr

## Architecture (Figure m5)

Display the locked architecture PNG. Optional regen runs `make_method_diagrams.py` (writes m1–m5).

In [ ]:
REGEN_ARCHITECTURE = False  # set True to rebuild method diagrams (incl. m5)

if REGEN_ARCHITECTURE:
    %run paper_figures/make_method_diagrams.py

show_figs("m5")

## Table Z — zero rate: daily vs weekly

Artifact: `ab_runs/weekly/zero_rate_daily_vs_weekly_locked800.json`

In [ ]:
pd.DataFrame([zr])

## Table 1 — Daily Direct-MH (primary)

Artifact: `ab_runs/weekly/daily_direct_mh60_locked800_s42.json`

This is the **primary** daily Results table (PAPER.md Table 1). Recursive bake-offs are appendix-only.

In [ ]:
daily = load_json(DEFAULT_PATHS["daily_direct_mh"])
hs = ["1", "7", "14", "28", "56", "60"]
rows = bakeoff_table(daily, horizons=hs)
print(format_markdown_table(rows, ["model", "method"] + [f"h{h}" for h in hs]))
pd.DataFrame(rows)

In [ ]:
rows_c = bakeoff_table(daily, horizons=hs, cum=True)
print("CumMAE")
print(format_markdown_table(rows_c, ["model", "method"] + [f"h{h}" for h in hs]))
pd.DataFrame(rows_c)

### Regenerate Figures D1–D2 (daily Direct-MH horizons)

In [ ]:
%run paper_figures/make_daily_direct_horizon_figures.py
show_figs("D1", "D2")

## Table D-S1 — Daily Direct-MH zone strata

Artifact: `ab_runs/weekly/strata_daily_direct_s42.json` (train mean-demand terciles).

In [ ]:
ds1 = strata_table("D-S1")
print(format_markdown_table(ds1, ["horizon", "zone", "deepsequence", "tsb", "lightgbm", "best"]))
pd.DataFrame(ds1)

## Table W — Weekly Direct-MH (primary)

Artifact: `ab_runs/weekly/weekly_mh8_locked800_s42.json`

In [ ]:
weekly = load_json(DEFAULT_PATHS["weekly_mh"])
rows = bakeoff_table(weekly, horizons=["1", "4", "8"])
print(format_markdown_table(rows, ["model", "method", "h1", "h4", "h8"]))
pd.DataFrame(rows)

In [ ]:
rows_c = bakeoff_table(weekly, horizons=["1", "4", "8"], cum=True)
print("CumMAE")
print(format_markdown_table(rows_c, ["model", "method", "h1", "h4", "h8"]))
pd.DataFrame(rows_c)

## Table L — Like-for-like Direct↔Direct

Matched leads: weekly \(h=1/4/8\) ≈ daily \(h=7/28/56\). Absolute IWMAE is **not** cross-grain comparable.

In [ ]:
ll = like_for_like_direct_table(weekly, daily)
pd.DataFrame(ll)

### Regenerate Figures W1–W3 (zero rate + weekly↔daily Direct-MH)

In [ ]:
%run paper_figures/make_weekly_daily_direct_compare.py
show_figs("W1", "W2", "W3")

## Tables W-S1 / W-S2 — Weekly Direct-MH zone strata

Artifact: `ab_runs/weekly/strata_weekly_direct_s42.json`
- **W-S1**: train mean-demand terciles
- **W-S2**: train zero-rate terciles

In [ ]:
ws1 = strata_table("W-S1")
print("### W-S1 mean-demand")
print(format_markdown_table(ws1, ["horizon", "zone", "deepsequence", "tsb", "lightgbm", "best"]))
pd.DataFrame(ws1)

In [ ]:
ws2 = strata_table("W-S2")
print("### W-S2 zero-rate")
print(format_markdown_table(ws2, ["horizon", "zone", "deepsequence", "tsb", "lightgbm", "best"]))
pd.DataFrame(ws2)

### Regenerate Figures D3 + W6 (Direct-MH strata bars)

In [ ]:
%run paper_figures/make_direct_strata_figures.py
show_figs("D3", "W6")

## Optional: Figures W4–W5 (weekly qualitative forecasts)

**Long / needs data.** Prefer displaying existing PNGs. Full regen retrains weekly Direct-MH dumps:

```bash
TF_USE_LEGACY_KERAS=1 .venv-test/bin/python paper_figures/make_forecast_weekly_plots.py --epochs 15 --max_skus 800 --seed 42
```

In [ ]:
REGEN_WEEKLY_FORECASTS = False  # True → long run; needs weekly panel / data

if REGEN_WEEKLY_FORECASTS:
    %run paper_figures/make_forecast_weekly_plots.py

show_figs("W4", "W5")

## Primary figure gallery (post-regen)

Inline check of all primary Results figures after the cells above.

In [ ]:
show_figs("D1", "D2", "D3", "W1", "W2", "W3", "W4", "W5", "W6", "m5")

## Appendix E only — recursive daily + multi-seed

**Not** primary Table 1. Kept for continuity with Appendix E (recursive DS/TST/LightGBM bake-off and five-seed IWMAE). Do not mix rankings with Direct-MH tables above.

In [ ]:
rec_rows = recursive_bakeoff_table()
print("Appendix E recursive IWMAE (not primary)")
print(format_markdown_table(rec_rows, ["model", "method", "h1", "h7", "h14", "h28", "h60"]))
pd.DataFrame(rec_rows)

In [ ]:
ms_rows = multiseed_iwmae_pivot()
pd.DataFrame(ms_rows).pivot(index="h", columns="model", values="mean")

## Optional: re-run Direct-MH bake-offs (long)

Default is **off**. Set `RUN_RERUN = True` only with Jubilant data + GPU/CPU budget. Fast path above already covers primary claims from locked artifacts.

In [ ]:
RUN_RERUN = False  # flip to True to re-train Direct-MH (hours on full 800)

if RUN_RERUN:
    import subprocess
    import shlex

    data = os.environ.get("DEEPSEQUENCE_DATA_DIR")
    if not data:
        raise RuntimeError("Set DEEPSEQUENCE_DATA_DIR")
    cmds = [
        f'''TF_USE_LEGACY_KERAS=1 .venv-test/bin/python -m deepsequence_hierarchical_attention.eval.weekly_mh '''
        f'''--data_dir ab_runs/weekly/panel_locked800 --feature_config feature_config_weekly.yaml '''
        f'''--sku_list ab_runs/recompare/sku_list_daily_data42.json --max_skus 800 '''
        f'''--horizon 8 --report_horizons 1,4,8 --models deepsequence,tsb,lightgbm '''
        f'''--epochs 15 --seed 42 --out_json ab_runs/weekly/weekly_mh8_locked800_s42.json''',
        f'''TF_USE_LEGACY_KERAS=1 .venv-test/bin/python -m deepsequence_hierarchical_attention.eval.weekly_mh '''
        f'''--data_dir {shlex.quote(data)} --feature_config feature_config.yaml '''
        f'''--dataset daily_direct_mh --sku_list ab_runs/recompare/sku_list_daily_data42.json '''
        f'''--max_skus 800 --horizon 60 --report_horizons 1,7,14,28,56,60 --mase_season 7 '''
        f'''--models deepsequence,tsb,lightgbm --epochs 15 --seed 42 '''
        f'''--out_json ab_runs/weekly/daily_direct_mh60_locked800_s42.json''',
    ]
    for c in cmds:
        print("RUN:", c)
        subprocess.check_call(c, shell=True)
else:
    print("Skipping re-run (RUN_RERUN=False). Artifacts already loaded above.")